# DDPG: Swinging Up a Pendulum using Gymnasium
Train a Deep Deterministic Policy Gradient (DDPG) agent to solve the `Pendulum-v1` environment using PyTorch and Gymnasium.

In [ ]:
!pip install gymnasium[box2d] torch matplotlib --quiet

In [ ]:
import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Create the Pendulum environment
env = gym.make("Pendulum-v1")

# Set seeds for reproducibility
seed = 42
env.reset(seed=seed)
torch.manual_seed(seed)
np.random.seed(seed)


In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim, max_action):
        super(Actor, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, action_dim),
            nn.Tanh()
        )
        self.max_action = max_action

    def forward(self, state):
        return self.max_action * self.net(state)


In [ ]:
class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super(Critic, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Linear(256, 1)
        )

    def forward(self, state, action):
        return self.net(torch.cat([state, action], dim=1))


In [ ]:
class ReplayBuffer:
    def __init__(self, max_size=100000):
        self.buffer = []
        self.max_size = max_size

    def add(self, experience):
        self.buffer.append(experience)
        if len(self.buffer) > self.max_size:
            self.buffer.pop(0)

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), size=batch_size)
        states, actions, rewards, next_states, dones = zip(*[self.buffer[i] for i in indices])
        return (
            torch.FloatTensor(states),
            torch.FloatTensor(actions),
            torch.FloatTensor(rewards).unsqueeze(1),
            torch.FloatTensor(next_states),
            torch.FloatTensor(dones).unsqueeze(1)
        )


In [ ]:
state_dim = env.observation_space.shape[0]
action_dim = env.action_space.shape[0]
max_action = float(env.action_space.high[0])

actor = Actor(state_dim, action_dim, max_action)
actor_target = Actor(state_dim, action_dim, max_action)
actor_target.load_state_dict(actor.state_dict())

critic = Critic(state_dim, action_dim)
critic_target = Critic(state_dim, action_dim)
critic_target.load_state_dict(critic.state_dict())

actor_optimizer = optim.Adam(actor.parameters(), lr=1e-4)
critic_optimizer = optim.Adam(critic.parameters(), lr=1e-3)

replay_buffer = ReplayBuffer()


In [ ]:
gamma = 0.99
tau = 0.005
batch_size = 64
exploration_noise = 0.1
episodes = 100
rewards_history = []

for episode in range(episodes):
    state, _ = env.reset()
    episode_reward = 0

    for step in range(200):
        state_tensor = torch.FloatTensor(state.reshape(1, -1))
        action = actor(state_tensor).detach().numpy()[0]
        action = action + exploration_noise * np.random.randn(action_dim)
        action = np.clip(action, -max_action, max_action)

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        replay_buffer.add((state, action, reward, next_state, done))

        state = next_state
        episode_reward += reward

        if len(replay_buffer.buffer) > batch_size:
            states, actions, rewards, next_states, dones = replay_buffer.sample(batch_size)

            with torch.no_grad():
                next_actions = actor_target(next_states)
                target_q = critic_target(next_states, next_actions)
                y = rewards + (1 - dones) * gamma * target_q

            critic_loss = nn.MSELoss()(critic(states, actions), y)
            critic_optimizer.zero_grad()
            critic_loss.backward()
            critic_optimizer.step()

            actor_loss = -critic(states, actor(states)).mean()
            actor_optimizer.zero_grad()
            actor_loss.backward()
            actor_optimizer.step()

            for param, target_param in zip(critic.parameters(), critic_target.parameters()):
                target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

            for param, target_param in zip(actor.parameters(), actor_target.parameters()):
                target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

        if done:
            break

    rewards_history.append(episode_reward)
    print(f"Episode {episode + 1}, Reward: {episode_reward:.2f}")


In [ ]:
plt.plot(rewards_history)
plt.xlabel("Episode")
plt.ylabel("Reward")
plt.title("DDPG on Pendulum-v1")
plt.grid()
plt.show()